In [ ]:
# === ノートブック共通の前処理 (llm_math パッケージの読み込み) ===
import sys
from pathlib import Path

# llm_math パッケージの候補パス
_candidates = [
    '.', 'src', '..', '../src',
    '/content/llm-math-book/src',
    '/content/llm-math-book',
    '/workspace/src',
    '/workspace',
]
# 親ディレクトリも候補に追加 (notebooks/ フォルダで実行する場合)
for p in Path.cwd().parents:
    _candidates.append(str(p / 'src'))
    _candidates.append(str(p))

for p in _candidates:
    if p and p not in sys.path and Path(p).exists():
        sys.path.insert(0, p)

# llm_math の import を試行
try:
    from llm_math import viz, bench, data
    _LLM_MATH_OK = True
except ImportError as e:
    _LLM_MATH_OK = False
    print(f"[注意] llm_math パッケージの読み込みエラー: {e}")
    print("  GitHub リポジトリを clone して colab_setup.sh を実行してください。")
# === 前処理ここまで ===


# Ch 12. トークナイザー

> **学習目標**
> - LLM
> - BPE Python
> - WordPiece, Unigram Language Model
> - トークナイザー(Byte-level BPE)

## 12.1

 :
- ** **: "Hello, world!" → ["Hello", ",", "world", "!"]
 - 問題: , / , OOV(Out-Of-Vocabulary)
- ** **: "Hello" → ['H', 'e', 'l', 'l', 'o']
 - 問題: ,
- ** **: "unhappiness" → ["un", "happiness"] ["un", "happ", "iness"]
 - **LLM **: 複雑度 ,

GPT-2: Byte-level BPE. BERT: WordPiece. T5: Unigram.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter, defaultdict

#    
text = "The quick brown fox jumps over the lazy dog."
word_tokens = text.lower().replace('.', ' .').split()
print(f"word : {word_tokens}")

# Character-level
char_tokens = list(text.lower())
print(f"Character-level: {char_tokens}")

#  (Concept )
subword_tokens = ['the', 'quick', 'brown', 'fox', 'jumps', 'over', 'the', 'lazy', 'dog', '.']
print(f"  (): {subword_tokens}")


## 12.2 Byte-Pair Encoding (BPE) — GPT-2

****:
1.
2. 複雑度 (2-gram)
3. 度

: 'low' (5), 'lower' (2), 'newest' (6), 'widest' (3)

複雑度 'e'+'s' 9 → → 'es'


In [ ]:
# BPE  
from collections import Counter

class BPE:
    def __init__(self, num_merges=100):
        self.num_merges = num_merges
        self.merges = []  # (a, b)   
        self.vocab = set()

    def _get_word_freqs(self, corpus):
        """word 複雑度 Computation."""
        word_freqs = Counter()
        for text in corpus:
            words = text.lower().split()
            for w in words:
                word_freqs[w] += 1
        return word_freqs

    def _get_pair_freqs(self, splits, word_freqs):
        """  複雑度 Computation."""
        pair_freqs = Counter()
        for word, split in splits.items():
            for i in range(len(split) - 1):
                pair = (split[i], split[i+1])
                pair_freqs[pair] += word_freqs[word]
        return pair_freqs

    def _merge(self, pair, splits):
        """ word pair Sum."""
        new_splits = {}
        a, b = pair
        for word, split in splits.items():
            new_split = []
            i = 0
            while i < len(split):
                if i < len(split) - 1 and split[i] == a and split[i+1] == b:
                    new_split.append(a + b)
                    i += 2
                else:
                    new_split.append(split[i])
                    i += 1
            new_splits[word] = new_split
        return new_splits

    def train(self, corpus):
        word_freqs = self._get_word_freqs(corpus)
        #  word Character-level Decomposition ( </w> Display)
        splits = {w: list(w) + ['</w>'] for w in word_freqs}
        #  
        self.vocab = set(c for w in word_freqs for c in w) | {'</w>'}

        for _ in range(self.num_merges):
            pair_freqs = self._get_pair_freqs(splits, word_freqs)
            if not pair_freqs:
                break
            best_pair = max(pair_freqs, key=pair_freqs.get)
            splits = self._merge(best_pair, splits)
            self.merges.append(best_pair)
            self.vocab.add(best_pair[0] + best_pair[1])

    def encode(self, word):
        """Training  word エンコード."""
        split = list(word) + ['</w>']
        for a, b in self.merges:
            i = 0
            new_split = []
            while i < len(split):
                if i < len(split) - 1 and split[i] == a and split[i+1] == b:
                    new_split.append(a + b)
                    i += 2
                else:
                    new_split.append(split[i])
                    i += 1
            split = new_split
        return split

#   Training
corpus = [
    "low low low low low",
    "lower lower newest newest newest",
    "widest widest newest newest",
]
bpe = BPE(num_merges=10)
bpe.train(corpus)

print("Training Sum :")
for i, (a, b) in enumerate(bpe.merges, 1):
    print(f"  {i}. {a} + {b} -> {a+b}")
print(f"\nVocabulary Size: {len(bpe.vocab)}")

# エンコード Test
for word in ['low', 'lowest', 'newer', 'widest']:
    tokens = bpe.encode(word)
    print(f"  {word:>10} -> {tokens}")


## 12.3 WordPiece — BERT

BPE ** **:

$$\text{score}(a, b) = \frac{\text{freq}(ab)}{\text{freq}(a) \cdot \text{freq}(b)}$$

BPE 複雑度 , WordPiece 複雑度 .
 , .


In [ ]:
# WordPiece  
class WordPiece:
    def __init__(self, num_merges=100):
        self.num_merges = num_merges
        self.merges = []

    def _get_pair_scores(self, splits, word_freqs):
        pair_freqs = Counter()
        char_freqs = Counter()
        for word, split in splits.items():
            for i in range(len(split) - 1):
                pair = (split[i], split[i+1])
                pair_freqs[pair] += word_freqs[word]
            for s in split:
                char_freqs[s] += word_freqs[word]
        scores = {p: f / (char_freqs[p[0]] * char_freqs[p[1]]) for p, f in pair_freqs.items()}
        return scores

    def _merge(self, pair, splits):
        new_splits = {}
        a, b = pair
        for word, split in splits.items():
            new_split = []
            i = 0
            while i < len(split):
                if i < len(split) - 1 and split[i] == a and split[i+1] == b:
                    new_split.append(a + b)
                    i += 2
                else:
                    new_split.append(split[i])
                    i += 1
            new_splits[word] = new_split
        return new_splits

    def train(self, corpus):
        word_freqs = Counter()
        for text in corpus:
            for w in text.lower().split():
                word_freqs[w] += 1
        # WordPiece ##     
        splits = {}
        for w in word_freqs:
            chars = list(w)
            splits[w] = [chars[0]] + ['##' + c for c in chars[1:]]
        for _ in range(self.num_merges):
            scores = self._get_pair_scores(splits, word_freqs)
            if not scores:
                break
            best = max(scores, key=scores.get)
            splits = self._merge(best, splits)
            self.merges.append(best)

wp = WordPiece(num_merges=10)
wp.train(corpus)
print("WordPiece Training Sum:")
for i, (a, b) in enumerate(wp.merges, 1):
    print(f"  {i}. {a} + {b} -> {a+b}")


## 12.4 Unigram Language Model — T5

BPE/WordPiece **bottom-up** ( ). Unigram **top-down** ( ).

1. = 
2. $P(t) = \text{count}(t) / \sum \text{count}$
3. $\mathcal{L} = -\sum_i \log P(\mathbf{x}_i) = -\sum_i \log \sum_{\text{seg}} \prod P(t)$
4.
5.

EM .


In [ ]:
# Unigram   ()
from itertools import combinations

class UnigramTokenizer:
    def __init__(self, vocab_size=100):
        self.vocab_size = vocab_size
        self.vocab = {}  # token -> prob

    def _get_candidates(self, word_freqs, max_len=10):
        candidates = Counter()
        for word, freq in word_freqs.items():
            for i in range(len(word)):
                for j in range(i+1, min(i+max_len+1, len(word)+1)):
                    candidates[word[i:j]] += freq
        return candidates

    def _segment(self, word, vocab):
        """Viterbi Optimal ."""
        n = len(word)
        # dp[i] = (best_log_prob, best_segmentation)
        dp = [(-float('inf'), None)] * (n + 1)
        dp[0] = (0, [])
        for i in range(1, n + 1):
            for j in range(max(0, i - 10), i):
                token = word[j:i]
                if token in vocab:
                    log_p = np.log(vocab[token])
                    if dp[j][0] + log_p > dp[i][0]:
                        dp[i] = (dp[j][0] + log_p, dp[j][1] + [token])
        return dp[n]

    def train(self, corpus):
        word_freqs = Counter()
        for text in corpus:
            for w in text.lower().split():
                word_freqs[w] += 1
        candidates = self._get_candidates(word_freqs)
        total = sum(candidates.values())
        self.vocab = {t: c / total for t, c in candidates.items()}

        # Iteration Loss Increase   
        while len(self.vocab) > self.vocab_size:
            #     Loss Increase Computation
            losses = {}
            for token in list(self.vocab.keys()):
                if len(token) == 1:  #   
                    continue
                #    Loss ()
                losses[token] = self.vocab[token]  #  Probability loss
            if not losses:
                break
            # Loss    
            sorted_tokens = sorted(losses.items(), key=lambda x: x[1])
            n_remove = max(1, len(self.vocab) - self.vocab_size)
            for token, _ in sorted_tokens[:n_remove]:
                del self.vocab[token]

#  
uni = UnigramTokenizer(vocab_size=30)
uni.train(corpus)
print(f"Unigram Vocabulary Size: {len(uni.vocab)}")
print(f" : {sorted(uni.vocab.items(), key=lambda x: -x[1])[:10]}")

#  
for word in ['lowest', 'newest']:
    log_p, seg = uni._segment(word, uni.vocab)
    print(f"  {word} -> {seg} (log_p={log_p:.4f})")


## 12.5 Byte-level BPE — GPT-2/3

問題: ( 11,172, ).
: **UTF-8 ** エンコード BPE .

- 256 → 256
- UNK(unknown) —
-

GPT-2 : 256 + 50,257 ≈ 50K


In [ ]:
# Byte-level BPE 
text = "Hello, !"
bytes_seq = text.encode('utf-8')
print(f"Original: {text}")
print(f"UTF-8 : {list(bytes_seq)}")
print(f" : {len(bytes_seq)} (  {len(text)})")
print("\n=>     . UNK .")

# HuggingFace tokenizers  
try:
    from tokenizers import Tokenizer
    from tokenizers.models import BPE as HFBPE
    from tokenizers.trainers import BpeTrainer
    from tokenizers.pre_tokenizers import Whitespace
except ImportError:
    print("\n[HuggingFace tokenizers  : pip install tokenizers]")

#  BPE Tokenizer (HuggingFace)
try:
    from tokenizers import Tokenizer
    from tokenizers.models import BPE as HFBPE
    from tokenizers.trainers import BpeTrainer
    from tokenizers.pre_tokenizers import Whitespace

    tokenizer = Tokenizer(HFBPE(unk_token="[UNK]"))
    tokenizer.pre_tokenizer = Whitespace()
    trainer = BpeTrainer(vocab_size=200, special_tokens=["[UNK]", "[PAD]", "[BOS]", "[EOS]"])

    #   Training
    from llm_math.data import load_tiny_shakespeare
    train_text = load_tiny_shakespeare(max_chars=5000)
    tokenizer.train_from_iterator([train_text], trainer)

    print(f"\nTraining Vocabulary Size: {tokenizer.get_vocab_size()}")
    # エンコード Test
    enc = tokenizer.encode("To be or not to be")
    print(f"'To be or not to be' -> {enc.tokens}")
    print(f"IDs: {enc.ids}")
except ImportError:
    print("\n[tokenizers  . pip install tokenizers ]")


## 12.6 [CPU/GPU ベンチマーク ④] トークナイザー 学習/エンコード 速度

トークナイザー CPU , 速度 .


In [ ]:
# トークナイザー ベンチマーク (CPU)
import time
from llm_math.data import load_tiny_shakespeare

#   
text = load_tiny_shakespeare(max_chars=100000)
print(f"ベンチマーク : {len(text):,}")

# 1.  BPE 学習 時間 
def bench_bpe_train(text, n_merges, repeat=3):
    times = []
    for _ in range(repeat):
        t0 = time.perf_counter()
        bpe = BPE(num_merges=n_merges)
        bpe.train([text])
        times.append(time.perf_counter() - t0)
    return np.mean(times), np.std(times)

print(f"\n{'merges':>8} | {'train (s)':>12}")
print("-" * 25)
for n in [10, 50, 100, 200]:
    m, s = bench_bpe_train(text, n, repeat=2)
    print(f"{n:>8} | {m:>10.4f}±{s:.4f}")

# 2. エンコード Speed
bpe_small = BPE(num_merges=50)
bpe_small.train([text[:5000]])

def bench_encode(bpe, text, n_words=1000):
    words = text.split()[:n_words]
    t0 = time.perf_counter()
    for w in words:
        bpe.encode(w)
    return (time.perf_counter() - t0) * 1000

t_ms = bench_encode(bpe_small, text, n_words=1000)
print(f"\n1000 エンコード: {t_ms:.2f}ms ({t_ms/1000:.3f}ms/)")
print("\n=> Tokenizer CPU .  の場合  .")
print("   HuggingFace tokenizers Rust Implementation  .")


## 12.7 性能

- → , モデル ,
- → , , 学習

LLM :
- GPT-2: 50,257
- GPT-3: 50,257
- LLaMA-2: 32,000
- LLaMA-3: 128,256
- GPT-4: ~100,000


In [ ]:
#   シーケンス長 比較 ()
#      
np.random.seed(0)
text_sample = load_tiny_shakespeare(max_chars=5000)

#    学習    比較
vocab_sizes = [30, 50, 100, 200, 500]
avg_tokens = []
for vs in vocab_sizes:
    bpe = BPE(num_merges=vs - 30)  #    30
    bpe.train([text_sample])
    #     
    sample_words = text_sample.split()[:100]
    total_tokens = sum(len(bpe.encode(w)) for w in sample_words)
    avg_tokens.append(total_tokens / len(sample_words))

plt.figure(figsize=(9, 5))
plt.plot(vocab_sizes, avg_tokens, 'o-', linewidth=2, markersize=8)
plt.xlabel('Vocabulary Size')
plt.ylabel('Mean   / word')
plt.title('Vocabulary Size Sequence Length Trade-off')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../figures/ch12_vocab_size_tradeoff.png', dpi=100, bbox_inches='tight')
plt.show()
print("=>   word   .  Embedding  .")


## 12.8 要点

| | | モデル |
|---|---|---|
| BPE | 複雑度 | GPT-2 |
| WordPiece | $\frac{\text{freq}(ab)}{\text{freq}(a)\text{freq}(b)}$ | BERT |
| Unigram | (top-down) | T5 |
| Byte-level BPE | BPE on bytes | GPT-2/3/4 |

## 演習問題

1. BPE , .
2. BPE WordPiece 学習 比較.
3. 100, 500, 1000 比較.
4. Byte-level BPE UNK .
5. HuggingFace `tokenizers` GPT-2 トークナイザー エンコード.

> 解答: `solutions/ch12_solutions.ipynb`
